# 05 - Resultado de negocio

Objetivo: transformar o modelo em decisao de negocio: ranking de clientes, faixas de risco, acao recomendada e ROI estimado.


In [ ]:
# Load final scored customers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

score = pd.read_csv('../data/processed/mod_churn_clientes_score_churn.csv')
score.head()


In [ ]:
# Summarize portfolio by risk band
risk_order = ['baixo', 'medio', 'alto', 'critico']
score['churn_real_bin'] = score['Churn_real'].map({'Yes': 1, 'No': 0})
resumo_risco = score.groupby('faixa_risco', observed=False).agg(
    clientes=('customerID', 'count'),
    churn_real=('churn_real_bin', 'sum'),
    taxa_churn_real=('churn_real_bin', 'mean'),
    prob_media_churn=('probabilidade_churn', 'mean'),
    mensalidade_media=('MonthlyCharges', 'mean')
).reindex(risk_order).reset_index()
resumo_risco


In [ ]:
# Plot risk distribution
ax = resumo_risco.plot(kind='bar', x='faixa_risco', y='clientes', legend=False, figsize=(7, 4), color='#2f6f9f')
ax.set_title('Clientes por faixa de risco')
ax.set_xlabel('Faixa de risco')
ax.set_ylabel('Clientes')
plt.tight_layout()
plt.show()


In [ ]:
# Simulate campaign ROI for different audience sizes
arpu = score['MonthlyCharges'].mean()
margin = 0.70
cost_unit = 10.0
uplift = 0.30
roi_rows = []
for k in [0.05, 0.10, 0.15, 0.20, 0.30]:
    n = int(round(len(score) * k))
    top = score.sort_values('probabilidade_churn', ascending=False).head(n)
    churners = int((top['Churn_real'] == 'Yes').sum())
    preserved = churners * arpu * margin * uplift
    cost = n * cost_unit
    roi_rows.append({'top_clientes': f'{int(k*100)}%', 'clientes': n, 'churners': churners, 'receita_preservada': preserved, 'custo': cost, 'roi': (preserved - cost) / cost})
roi = pd.DataFrame(roi_rows)
roi
